> **Hands-on workshop for the AI-Driven DevOps (AIDD) program, School of Computer Science and Engineering, Jain (Deemed-to-be University), Bengaluru**

# 02 · AWS essentials: accounts, IAM, regions, CLI

**CampusMate v0**  ·  ⏱️ Time budget: **10 min theory + 20 min hands-on**

*All student, course, policy and event data in this workshop is fictional.*

## Why this matters

CampusMate will run on Amazon Web Services (AWS). To deploy anything you must know *which account* you are in, *who* you are, *where* (region) things run, and how to read a 'permission denied' message without panicking. Your own user has admin access, so today is just about knowing where you are.

### What you'll do
- Confirm your identity with `aws sts get-caller-identity`
- Set and check your region
- Run the one-command setup check
- Then we take a first look at Amazon Bedrock in the AWS console

## Settings

In [ ]:
# ---- Step 0: make sure the workshop libraries are in THIS notebook's Python (macOS, Windows, Linux) ----
import sys, subprocess, importlib, importlib.util
if sys.version_info < (3, 10):
    raise SystemExit(f"This notebook runs Python {sys.version.split()[0]}. Choose the .venv kernel (Python 3.10+) and run again.")
if any(importlib.util.find_spec(m) is None for m in ("strands", "strands_tools", "bedrock_agentcore", "mcp_proxy_for_aws", "boto3", "dotenv")):
    print("Installing the workshop libraries into", sys.executable, "(1-2 minutes, only the first time)...")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check",
                               "strands-agents==1.64.0", "strands-agents-tools==0.9.0", "bedrock-agentcore==1.25.4",
                               "mcp-proxy-for-aws==1.8.0", "boto3==1.43.126", "botocore==1.43.126",
                               "flask==3.1.3", "python-dotenv==1.2.4", "requests==2.34.2"])
        importlib.invalidate_caches()
    except subprocess.CalledProcessError:
        raise SystemExit("Could not install here. Choose the .venv kernel (VS Code: Select Kernel -> .venv; "
                         "Jupyter: start it from a terminal after activating .venv), then run this cell again.")
print("Libraries ready in:", sys.executable)

# Settings for this lab
import sys, json, time, uuid, pathlib

REGION = "us-east-1"                                 # the AWS region we use in this workshop
MODEL_ID = "us.amazon.nova-2-lite-v1:0"              # Amazon Nova 2 Lite (CampusMate's brain)
CHEAP_MODEL_ID = "us.amazon.nova-2-lite-v1:0"        # same model for quick experiments

ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
DATA_DIR = ROOT / "data"                             # the fictional campus data
print("Region:", REGION, "| Model:", MODEL_ID)

## Accounts, IAM and regions in 4 lines

- **AWS account**: a container for everything you create and the bill for it. You each have your own.
- **IAM (Identity and Access Management)**: *who* can do *what* on *which* resource. Your login is an **IAM user**; services use **IAM roles**.
- **Policy**: a JSON document listing allowed actions, e.g. `bedrock:InvokeModel`. No allow = denied.
- **Region**: a physical AWS location like `us-east-1` (N. Virginia) or `ap-south-1` (Mumbai). Not every service is in every region.

🏫 **Analogy:** the account is the college; IAM is the ID-card system; a policy is the list of rooms your card opens;
a region is which campus you're on.

## Why we use us-east-1

Amazon Bedrock **AgentCore** (Runtime, Memory, Gateway, Policy, Evaluations) was verified in **us-east-1** for this workshop.
The Nova inference profile we use starts with `us.` because they belong to US regions. If you move to Mumbai (`ap-south-1`) you'd use
`apac.` profiles instead, but check AgentCore availability first.

## The AWS CLI

The **CLI (Command Line Interface)** lets you talk to AWS from a terminal. `aws configure` stores your access key in
`~/.aws/credentials` (outside this repo, never in `.env`, never in git).
**Windows:** the same file lives at `C:\Users\<you>\.aws\credentials`. The `aws` commands are the same.

## 🔬 Live check: who am I?

The `!` runs a terminal command from the notebook.

In [ ]:
!aws sts get-caller-identity
!aws configure get region

In [ ]:
import boto3
me = boto3.client("sts").get_caller_identity()
print("Account :", me["Account"])
print("I am    :", me["Arn"])
print("Region  :", REGION, "(set in the Settings cell)")

## 🔬 Run the full setup check

This is the same script you ran during pre-work. Every line should say PASS.

In [ ]:
!{sys.executable} ../scripts/check_setup.py

## ✅ Checkpoint

You are on track if:
- `get-caller-identity` shows your account and your IAM user
- `check_setup.py` ends with 'All checks passed'

**Stuck or behind?** Copy from `scripts/check_setup.py` and keep going. That's what it's there for.

## 🛠️ Common errors

| You see | Why | Fix |
|---|---|---|
| `Unable to locate credentials` | `aws configure` not done | Run `aws configure`, paste your keys, region `us-east-1` |
| `ExpiredToken` / `InvalidClientTokenId` | Keys expired or mistyped | Create a new access key in IAM and run `aws configure` again |
| `'aws' is not recognized` | AWS CLI v2 not installed or terminal not restarted | Install AWS CLI v2 (SETUP.md) and reopen the terminal |